# Generate Insights Brief Descriptions

This notebook:
1. Reads `Insights_brief_description_To_be_generated.xlsx` (expected in the same directory as this notebook / working directory)
2. Connects to GitLab and fetches issue title + description for each unique `issue_id`
3. Calls an LLM (OpenAI / Azure OpenAI) to produce a **very brief summary** per `rule_name`
4. Writes results back to an output Excel file

**Supports both local execution and Airflow PythonOperator.**

## 1. Imports & Logging

In [ ]:
"""
generate_insights_brief_descriptions.ipynb

Dependencies:
    pip install python-gitlab pandas openpyxl openai

Environment variables (local runs, all optional — getpass prompt used as fallback):
    GENESIS_DDLC_IKG_GIT_SECRET  – GitLab private token
    OPENAI_API_KEY               – OpenAI / Azure-OpenAI API key (prompted if absent)

Airflow variables / connections:
    GENESIS_DDLC_IKG_GIT_SECRET  – Airflow Variable with GitLab token
    STAAT-DS-OPENAI-LLM          – Airflow Connection (host=base_url, password=api_key)
"""

from __future__ import annotations

import getpass
import logging
import os
import textwrap
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

logger.info("Imports loaded successfully.")

## 2. Configuration

In [ ]:
# ---------------------------------------------------------------------------
# GitLab
# ---------------------------------------------------------------------------
GITLAB_URL = "https://devcloud.ubs.net"
PROJECT_PATH = (
    "ubs/gwma/smart-technology-and-analytics/staat-data-science/"
    "staat-ds-insights-cl/commons/staat-ds-insights-home"
)
GITLAB_TOKEN_VAR = "GENESIS_DDLC_IKG_GIT_SECRET"   # Airflow Variable name / env-var name

# ---------------------------------------------------------------------------
# LLM
# ---------------------------------------------------------------------------
# Model for brief_description (concise body text)
MODEL_NAME        = "gpt-4.1-mini"
# Model for insight_title (ultra-short headline) — uses the full gpt-4.1
TITLE_MODEL_NAME  = "gpt-4.1"
MAX_TOKENS        = 9000   # max total tokens per request
TEMPERATURE       = 0.1
OPENAI_BASE_URL   = "https://cirruspl-staat-ste-dev-ai.openai.azure.com/"

# ---------------------------------------------------------------------------
# I/O
# ---------------------------------------------------------------------------
# Place the xlsx next to this notebook (or set an absolute path here)
INPUT_FILENAME  = "Insights_brief_description_To_be_generated.xlsx"
OUTPUT_FILENAME = "Insights_brief_description_generated.xlsx"

logger.info("Configuration set.")

## 3. Airflow Detection & Credential Helpers

In [ ]:
# ---------------------------------------------------------------------------
# Airflow detection  (mirrors pattern used in release_gitlab_report.py)
# ---------------------------------------------------------------------------
try:
    from airflow.models import Variable  # type: ignore
    _HAS_AIRFLOW = True
except Exception:
    Variable = None  # type: ignore
    _HAS_AIRFLOW = False


def is_running_in_airflow() -> bool:
    """True only when actually executing inside an Airflow task context."""
    return _HAS_AIRFLOW and bool(os.environ.get("AIRFLOW_CTX_DAG_ID"))


# ---------------------------------------------------------------------------
# GitLab token
# ---------------------------------------------------------------------------
def get_private_token(allow_prompt: bool = True) -> str:
    """Resolve GitLab private token.

    Airflow  → Airflow Variable ``GENESIS_DDLC_IKG_GIT_SECRET``
    Local    → env var ``GENESIS_DDLC_IKG_GIT_SECRET`` or interactive prompt
    """
    if is_running_in_airflow():
        token = Variable.get(GITLAB_TOKEN_VAR)
        if not token:
            raise RuntimeError(f"Airflow Variable '{GITLAB_TOKEN_VAR}' must be set.")
        logger.info("GitLab token resolved from Airflow Variable.")
        return token

    token = os.environ.get(GITLAB_TOKEN_VAR, "")
    if token:
        logger.info("GitLab token resolved from environment variable.")
        return token
    if allow_prompt:
        return getpass.getpass(f"Enter GitLab private token ({GITLAB_TOKEN_VAR}): ")
    raise RuntimeError("GitLab token not found. Set env var or enable interactive prompt.")


# ---------------------------------------------------------------------------
# OpenAI / Azure-OpenAI credentials
# ---------------------------------------------------------------------------
def get_openai_credentials(allow_prompt: bool = True) -> Tuple[str, str]:
    """Return (api_key, base_url).

    Resolution order
    ----------------
    Airflow  → Airflow Connection ``STAAT-DS-OPENAI-LLM``
               (password = api_key, host = base_url)
    Local    → env var ``OPENAI_API_KEY``  (base_url = OPENAI_BASE_URL constant)
             → interactive getpass prompt if env var is absent and allow_prompt=True
    """
    if is_running_in_airflow():
        try:
            from airflow.models import Connection  # type: ignore
            conn = Connection.get_connection_from_secrets("STAAT-DS-OPENAI-LLM")
            api_key  = conn.password
            base_url = conn.host or OPENAI_BASE_URL
            if not api_key:
                raise RuntimeError("OpenAI API key not found in Airflow connection 'STAAT-DS-OPENAI-LLM'.")
            logger.info("OpenAI credentials resolved from Airflow connection 'STAAT-DS-OPENAI-LLM'.")
            return api_key, base_url
        except Exception as exc:
            logger.error("Failed to retrieve OpenAI connection from Airflow: %s", exc)
            raise RuntimeError(f"Could not retrieve OpenAI credentials from Airflow: {exc}") from exc

    # --- Local run ---
    # 1. Try environment variable first (zero-friction for CI / power users)
    api_key = os.environ.get("OPENAI_API_KEY", "").strip()
    if api_key:
        logger.info("OpenAI API key resolved from environment variable OPENAI_API_KEY.")
        return api_key, OPENAI_BASE_URL

    # 2. Fall back to interactive prompt (mirrors getpass pattern in release_gitlab_report.py)
    if allow_prompt:
        api_key = getpass.getpass("Enter OpenAI / Azure-OpenAI API key: ").strip()
        if not api_key:
            raise RuntimeError("OpenAI API key cannot be empty.")
        logger.info("OpenAI API key provided via interactive prompt.")
        return api_key, OPENAI_BASE_URL

    raise RuntimeError(
        "OpenAI API key not found. "
        "Set the OPENAI_API_KEY environment variable or enable allow_prompt=True."
    )


logger.info("Credential helpers defined. Running in Airflow: %s", is_running_in_airflow())

## 4. Read Input Spreadsheet

In [ ]:
def resolve_input_path() -> Path:
    """Return absolute path to the input file.

    When running inside Airflow the DAG file lives in AIRFLOW_HOME/dags;
    we expect the xlsx in the same folder. Locally it is next to this notebook.
    """
    # Try the directory that contains this notebook / script first
    candidates = [
        Path(os.path.abspath("__file__")).parent / INPUT_FILENAME  if "__file__" in dir() else None,
        Path.cwd() / INPUT_FILENAME,
        Path(os.environ.get("AIRFLOW_HOME", "")) / "dags" / INPUT_FILENAME,
    ]
    for path in candidates:
        if path and path.exists():
            return path
    raise FileNotFoundError(
        f"Input file '{INPUT_FILENAME}' not found. "
        "Place it in the same directory as this notebook."
    )


def load_input_df(path: Path) -> pd.DataFrame:
    """Load the input xlsx/csv and validate required columns."""
    suffix = path.suffix.lower()
    if suffix in (".xlsx", ".xls"):
        df = pd.read_excel(path, dtype=str)
    else:
        df = pd.read_csv(path, dtype=str)

    required_cols = {"rule_name", "issue_id"}
    missing = required_cols - set(df.columns.str.strip().str.lower())
    if missing:
        raise ValueError(f"Input file is missing required columns: {missing}")

    df.columns = df.columns.str.strip().str.lower()
    df["issue_id"] = df["issue_id"].str.strip()
    df["rule_name"] = df["rule_name"].str.strip()
    logger.info("Loaded %d rows from '%s'", len(df), path.name)
    return df


input_path = resolve_input_path()
df_input = load_input_df(input_path)
df_input.head()

## 5. Connect to GitLab & Fetch Issue Details

In [ ]:
import gitlab  # pip install python-gitlab


def connect_gitlab(token: str) -> gitlab.Gitlab:
    gl = gitlab.Gitlab(GITLAB_URL, private_token=token)
    gl.auth()
    logger.info("Connected to GitLab: %s", GITLAB_URL)
    return gl


def fetch_issue_details(
    gl: gitlab.Gitlab,
    project_path: str,
    issue_ids: List[str],
) -> Dict[str, Dict]:
    """Fetch title and description for each issue IID.

    Returns a dict keyed by issue_id (string) →
        {"title": str, "description": str}
    """
    project = gl.projects.get(project_path)
    logger.info("Opened project: %s", project.path_with_namespace)

    result: Dict[str, Dict] = {}
    unique_ids = sorted(set(issue_ids), key=lambda x: int(x) if x.isdigit() else 0)

    for iid in unique_ids:
        try:
            issue = project.issues.get(int(iid))
            result[iid] = {
                "title": issue.title or "",
                "description": issue.description or "",
            }
            logger.info("  Fetched issue #%s: %s", iid, issue.title)
        except Exception as exc:
            logger.warning("  Could not fetch issue #%s: %s", iid, exc)
            result[iid] = {"title": "", "description": ""}

    logger.info("Fetched details for %d unique issues.", len(result))
    return result


# Resolve credentials and connect
private_token = get_private_token(allow_prompt=True)
gl = connect_gitlab(private_token)

# Fetch all unique issue IDs present in the spreadsheet
issue_ids = df_input["issue_id"].dropna().unique().tolist()
issue_details: Dict[str, Dict] = fetch_issue_details(gl, PROJECT_PATH, issue_ids)

# Preview
for iid, details in list(issue_details.items())[:3]:
    print(f"\n--- Issue #{iid} ---")
    print(f"Title      : {details['title']}")
    print(f"Description: {details['description'][:200]}...")

## 6. Build LLM Prompts & Generate Brief Description + Insight Title

In [ ]:
from openai import OpenAI  # pip install openai


# ---------------------------------------------------------------------------
# brief_description  — gpt-4.1-mini
# A concise plain-English explanation of what the rule does (max 40 words).
# ---------------------------------------------------------------------------
def build_description_prompt(rule_name: str, title: str, description: str) -> str:
    desc_short = textwrap.shorten(description or "", width=4000, placeholder=" …")
    return (
        "You are an expert release manager writing concise insight descriptions.\n\n"
        f"Rule name  : {rule_name}\n"
        f"Story title: {title}\n"
        f"Story description:\n{desc_short}\n\n"
        "Write a VERY brief (max 40 words) plain-English description of what this "
        "rule does, based on the story above. Focus on the business action or insight "
        "surfaced. Do not mention the rule_name verbatim. Do not invent details."
    )


def generate_brief_description(client: OpenAI, rule_name: str, title: str, description: str) -> str:
    """Generate brief_description using gpt-4.1-mini."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": build_description_prompt(rule_name, title, description)}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
    )
    return response.choices[0].message.content.strip()


# ---------------------------------------------------------------------------
# insight_title  — gpt-4.1
# A very brief human-readable title/name for the rule (3-6 words max).
# Derived purely from the rule_name — no GitLab content needed.
# ---------------------------------------------------------------------------
def build_title_prompt(rule_name: str) -> str:
    return (
        "You are naming business insight rules for a financial advisory platform.\n\n"
        f"Rule name (snake_case): {rule_name}\n\n"
        "Convert this rule name into a very brief, human-readable title (3 to 6 words). "
        "Use title case. Replace underscores with spaces and expand abbreviations where clear. "
        "Output only the title, nothing else."
    )


def generate_insight_title(client: OpenAI, rule_name: str) -> str:
    """Generate insight_title using gpt-4.1."""
    response = client.chat.completions.create(
        model=TITLE_MODEL_NAME,
        messages=[{"role": "user", "content": build_title_prompt(rule_name)}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
    )
    return response.choices[0].message.content.strip()


# Initialise a single shared OpenAI client (both models share the same endpoint)
api_key, base_url = get_openai_credentials()
client = OpenAI(api_key=api_key, base_url=base_url)
logger.info("OpenAI client ready.  description model=%s  title model=%s", MODEL_NAME, TITLE_MODEL_NAME)

In [ ]:
# ---------------------------------------------------------------------------
# Iterate over every row — generate insight_title + brief_description
# ---------------------------------------------------------------------------
insight_titles:     List[str] = []
brief_descriptions: List[str] = []

for idx, row in df_input.iterrows():
    rule_name = str(row.get("rule_name", "")).strip()
    issue_id  = str(row.get("issue_id",  "")).strip()

    details     = issue_details.get(issue_id, {})
    title       = details.get("title",       "")
    description = details.get("description", "")

    # ---- insight_title (gpt-4.1) — derived from rule_name only ----
    try:
        ins_title = generate_insight_title(client, rule_name)
        insight_titles.append(ins_title)
        logger.info("Row %d | rule=%-50s | insight_title: %s", idx, rule_name, ins_title)
    except Exception as exc:
        logger.error("Row %d | rule=%s | insight_title error: %s", idx, rule_name, exc)
        insight_titles.append("")

    # ---- brief_description (gpt-4.1-mini) — needs issue content ----
    if not title and not description:
        logger.warning("Row %d | rule=%s | issue #%s has no content — skipping brief_description.",
                       idx, rule_name, issue_id)
        brief_descriptions.append("")
        continue

    try:
        desc = generate_brief_description(client, rule_name, title, description)
        brief_descriptions.append(desc)
        logger.info("Row %d | rule=%-50s | brief_description: %d chars", idx, rule_name, len(desc))
    except Exception as exc:
        logger.error("Row %d | rule=%s | brief_description error: %s", idx, rule_name, exc)
        brief_descriptions.append("")

df_input["insight_title"]    = insight_titles
df_input["brief_description"] = brief_descriptions
logger.info("Done. %d rows processed.", len(df_input))
df_input[["rule_name", "issue_id", "insight_title", "brief_description"]].head(10)

## 7. Export Results

In [ ]:
output_path = Path.cwd() / OUTPUT_FILENAME

df_input.to_excel(output_path, index=False, engine="openpyxl")
logger.info("Results written to: %s", output_path)
print(f"\n✅ Output saved to: {output_path}")

## 8. Airflow-Compatible Entry Point

Wrap the end-to-end logic into a single callable so that an Airflow `PythonOperator` can invoke it without any interactive prompts.

In [ ]:
def generate_insights_brief_descriptions(
    private_token: Optional[str] = None,
    output_path_override: Optional[str] = None,
    **kwargs,  # absorbs Airflow context kwargs
) -> str:
    """End-to-end pipeline — callable from Airflow PythonOperator or locally.

    Parameters
    ----------
    private_token:
        GitLab private token. When None, resolved from Airflow Variable or env/prompt.
    output_path_override:
        Absolute path for the output file. Defaults to cwd / OUTPUT_FILENAME.

    Returns
    -------
    str  Absolute path of the generated output Excel file.
    """
    # ---- 1. Input data -------------------------------------------------------
    _input_path = resolve_input_path()
    _df = load_input_df(_input_path)

    # ---- 2. GitLab -----------------------------------------------------------
    _token   = private_token or get_private_token(allow_prompt=False)
    _gl      = connect_gitlab(_token)
    _ids     = _df["issue_id"].dropna().unique().tolist()
    _details = fetch_issue_details(_gl, PROJECT_PATH, _ids)

    # ---- 3. LLM — single client, two models ----------------------------------
    _api_key, _base_url = get_openai_credentials(allow_prompt=False)
    _client = OpenAI(api_key=_api_key, base_url=_base_url)

    _insight_titles:     List[str] = []
    _brief_descriptions: List[str] = []

    for _, row in _df.iterrows():
        rule  = str(row.get("rule_name", "")).strip()
        iid   = str(row.get("issue_id",  "")).strip()
        det   = _details.get(iid, {})
        ttl   = det.get("title",       "")
        dsc   = det.get("description", "")

        # insight_title — gpt-4.1, rule_name only
        try:
            _insight_titles.append(generate_insight_title(_client, rule))
        except Exception as exc:
            logger.error("insight_title error for rule %s: %s", rule, exc)
            _insight_titles.append("")

        # brief_description — gpt-4.1-mini, needs issue content
        if not ttl and not dsc:
            _brief_descriptions.append("")
            continue
        try:
            _brief_descriptions.append(generate_brief_description(_client, rule, ttl, dsc))
        except Exception as exc:
            logger.error("brief_description error for rule %s: %s", rule, exc)
            _brief_descriptions.append("")

    _df["insight_title"]    = _insight_titles
    _df["brief_description"] = _brief_descriptions

    # ---- 4. Output -----------------------------------------------------------
    _out = Path(output_path_override) if output_path_override else Path.cwd() / OUTPUT_FILENAME
    _df.to_excel(_out, index=False, engine="openpyxl")
    logger.info("✅ Output written to: %s", _out)
    return str(_out)


# ---------------------------------------------------------------------------
# Local execution guard
# ---------------------------------------------------------------------------
if __name__ == "__main__" and not is_running_in_airflow():
    result_path = generate_insights_brief_descriptions()
    print(f"Done. Output: {result_path}")

---
### Airflow DAG snippet

```python
# In your DAG file:
from airflow.operators.python import PythonOperator
from generate_insights_brief_descriptions import generate_insights_brief_descriptions

generate_task = PythonOperator(
    task_id="generate_insights_brief_descriptions",
    python_callable=generate_insights_brief_descriptions,
    op_kwargs={
        # token resolved automatically from Airflow Variable when omitted
    },
    dag=dag,
)
```